# 01 - Exploratory Data Analysis

This notebook performs exploratory data analysis on the clickstream data.

In [ ]:
# =============================================================================
# Import Required Libraries
# =============================================================================
# Data manipulation and numerical computing
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path for importing custom modules
import sys
sys.path.append('..')

# Custom utility functions
from src.load_data import load_raw_data
from src.utils import set_seed, memory_usage

# Set random seed for reproducibility
set_seed(42)

# Enable inline plotting in Jupyter
%matplotlib inline

## Load Raw Data

In [ ]:
# =============================================================================
# Load and Standardize Raw Clickstream Data
# =============================================================================
import sys, importlib
sys.path.append("..")

# Reload modules to pick up any code changes during development
import src.load_data
importlib.reload(src.load_data)

from src.load_data import load_raw_data, standardize_columns

# Load raw events from CSV (semicolon-separated format)
df = load_raw_data("../data/raw/events.csv")

# Standardize column names and create event_time from date components
# Also derives event_type: last click per session = "purchase", others = "view"
df = standardize_columns(df)

# Preview data structure
print(f"Dataset shape: {df.shape}")
df.head()
df.columns.tolist()

## Data Overview

In [ ]:
# =============================================================================
# Event Type Distribution
# =============================================================================
# Count occurrences of each event type (view vs purchase)
# This shows the class imbalance in our dataset
df["event_type"].value_counts()

In [ ]:
# =============================================================================
# Event Types Per Session (Sample View)
# =============================================================================
# Examine event patterns within individual sessions
# Shows how views and purchases are distributed across sessions
df.groupby("session_id")["event_type"].value_counts().head(10)

## Data Visualization

In [ ]:
# =============================================================================
# Session Length vs Purchase Outcome
# =============================================================================
# Analyze whether session length (number of events) correlates with purchase behavior
# Hypothesis: Users who purchase may have different browsing patterns

import seaborn as sns
import matplotlib.pyplot as plt

# Aggregate session-level statistics
session_summary = (
    df.groupby("session_id")
      .agg(
          n_events=("event_type", "count"),                           # Total clicks in session
          purchased=("event_type", lambda x: (x == "purchase").any()) # Did session end in purchase?
      )
      .reset_index()
)

# Visualize session length distribution by purchase outcome
plt.figure(figsize=(6, 4))
sns.boxplot(
    data=session_summary,
    x="purchased",
    y="n_events"
)
plt.xticks([0, 1], ["No Purchase", "Purchase"])
plt.title("Session Length vs Purchase Outcome")
plt.ylabel("Number of Events")
plt.xlabel("")
plt.show()

In [ ]:
# =============================================================================
# Price Distribution: Views vs Purchases
# =============================================================================
# Compare price distributions between viewed and purchased products
# Helps understand if price sensitivity affects purchase decisions

plt.figure(figsize=(6, 4))
sns.kdeplot(
    data=df[df["event_type"].isin(["view", "purchase"])],
    x="price",
    hue="event_type",
    common_norm=False  # Normalize each distribution independently
)
plt.title("Price Distribution: Views vs Purchases")
plt.xlabel("Price")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Compute session length
session_stats = (
    df.groupby("session_id")
      .agg(
          session_length=("event_type", "count"),
          purchased=("event_type", lambda x: (x == "purchase").any())
      )
)

plt.figure(figsize=(8, 4))

plt.hist(
    session_stats[session_stats["purchased"]]["session_length"],
    bins=40,
    alpha=0.7,
    label="Purchased",
    log=True
)

plt.hist(
    session_stats[~session_stats["purchased"]]["session_length"],
    bins=40,
    alpha=0.7,
    label="Not Purchased",
    log=True
)

plt.title("Session Length Distribution by Purchase Outcome")
plt.xlabel("Number of Events in Session")
plt.ylabel("Number of Sessions (log scale)")
plt.legend()
plt.tight_layout()
plt.show()


Sessions that end in purchase tend to be significantly longer, indicating deeper engagement before conversion. This validates the use of session-level behavioral aggregates as predictive features.